# PRE-SMOTE — Sinh mẫu tổng hợp cho 3 lớp thiểu số (IEC 104)

Notebook này thực hiện đúng bước tiền xử lý mô tả ở **mục 4.3.1** của luận văn:

- Chỉ áp dụng SMOTE cho các lớp có **dưới 500 mẫu** trên tập **train**: `attackfree`, `floodattack`, `mitmattack`.
- Mỗi lớp trong 3 lớp trên được nâng lên **đúng 500 mẫu**.
- 5 lớp còn lại (`dosattack`, `fuzzyattack`, `iec104starvationattack`, `ntpddosattack`, `portscanattack`) **giữ nguyên**, không sinh thêm cũng không loại bớt.
- **Không đụng vào tập validation/test** — SMOTE chỉ áp dụng trên `X_train`/`y_train` để đảm bảo phép so sánh công bằng với TabNet gốc (đúng như luận văn đã nêu).
- Với lớp có số mẫu gốc quá ít (ví dụ `mitmattack`), `k_neighbors` của SMOTE sẽ **tự động hạ xuống** cho vừa với số mẫu hiện có, tránh lỗi runtime.

**Input:** `X_train.csv`, `y_train.csv`, `label_encoder.pkl` (đã có sẵn từ bước tiền xử lý ban đầu, dùng chung với `TABNET.ipynb`).

**Output:**
- `X_train_smote.csv`, `y_train_smote.csv` — tập train sau khi SMOTE, dùng cho notebook huấn luyện `train_tabnet_smote.ipynb`.
- `smote_class_distribution.json` — bảng phân bố lớp trước/sau SMOTE (đối chiếu Bảng 4.2 trong luận văn).


In [ ]:
# Cài đặt nếu máy chưa có imbalanced-learn (chỉ cần chạy 1 lần)
# !pip install imbalanced-learn


In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
from collections import Counter
from imblearn.over_sampling import SMOTE

pd.set_option("display.width", 120)


In [ ]:
# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN — đổi lại cho khớp máy bạn (giống TABNET.ipynb)
# ==============================================================================
data_dir = "../data/processed/"

RANDOM_STATE = 42          # khớp seed=42 dùng trong TABNET.ipynb (tabnet_params['seed'])
TARGET_COUNT = 500         # số mẫu mục tiêu cho các lớp thiểu số, đúng Bảng 4.2
MIN_CLASS_SIZE_TO_TOUCH = 500  # chỉ SMOTE các lớp có số mẫu train < ngưỡng này
DEFAULT_K_NEIGHBORS = 5    # k_neighbors mặc định của SMOTE


In [ ]:
# ==============================================================================
# 2. TẢI DỮ LIỆU GỐC (giống hệt cell tải dữ liệu trong TABNET.ipynb)
# ==============================================================================
X_train = pd.read_csv(data_dir + "X_train.csv")
y_train = pd.read_csv(data_dir + "y_train.csv").values.ravel()

with open(data_dir + "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

label_names = label_encoder.classes_
feature_names = X_train.columns.tolist()

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Số lớp: {len(label_names)}, số features: {len(feature_names)}")


In [ ]:
# ==============================================================================
# 3. PHÂN BỐ LỚP TRƯỚC SMOTE
# ==============================================================================
counts_before = Counter(y_train)

print("Phân bố lớp trên tập train TRƯỚC SMOTE:")
print("-" * 45)
for idx, name in enumerate(label_names):
    n = counts_before.get(idx, 0)
    flag = "  <-- sẽ SMOTE lên 500" if n < MIN_CLASS_SIZE_TO_TOUCH else ""
    print(f"  {name:<25} {n:>10,d}{flag}")
print("-" * 45)
print(f"  {'TỔNG':<25} {sum(counts_before.values()):>10,d}")


In [ ]:
# ==============================================================================
# 4. ÁP DỤNG SMOTE — chỉ cho các lớp dưới 500 mẫu, nâng lên đúng 500
#
# Áp dụng TUẦN TỰ từng lớp thiểu số (thay vì gọi SMOTE 1 lần cho cả 3 lớp),
# vì mỗi lớp có thể cần k_neighbors khác nhau tùy số mẫu gốc hiện có.
# Cách làm này tương đương về mặt kết quả nhưng an toàn hơn khi 1 trong các
# lớp có số mẫu gốc rất thấp (ví dụ mitmattack).
# ==============================================================================
X_res, y_res = X_train.values.copy(), y_train.copy()

minority_classes = [idx for idx, name in enumerate(label_names)
                     if counts_before.get(idx, 0) < MIN_CLASS_SIZE_TO_TOUCH]

print(f"Các lớp sẽ được SMOTE: {[label_names[i] for i in minority_classes]}\n")

for cls_idx in minority_classes:
    n_current = int(np.sum(y_res == cls_idx))

    if n_current >= TARGET_COUNT:
        # Trường hợp hiếm: lớp đã đủ 500 sau vòng lặp trước (không xảy ra ở đây
        # vì các lớp thiểu số không giao nhau), bỏ qua để tránh lỗi.
        continue

    # k_neighbors không được >= số mẫu gốc của lớp đó -> tự động hạ xuống
    k = min(DEFAULT_K_NEIGHBORS, max(1, n_current - 1))

    if n_current < 2:
        print(f"  ⚠ Lớp {label_names[cls_idx]} có {n_current} mẫu — không đủ để SMOTE, bỏ qua.")
        continue

    print(f"  {label_names[cls_idx]:<25} {n_current:>6,d} mẫu gốc -> "
          f"target {TARGET_COUNT}, k_neighbors = {k}")

    smote = SMOTE(
        sampling_strategy={cls_idx: TARGET_COUNT},
        k_neighbors=k,
        random_state=RANDOM_STATE,
    )
    X_res, y_res = smote.fit_resample(X_res, y_res)

print("\n✓ Hoàn tất SMOTE.")


In [ ]:
# ==============================================================================
# 5. PHÂN BỐ LỚP SAU SMOTE — đối chiếu với Bảng 4.2 trong luận văn
# ==============================================================================
counts_after = Counter(y_res)

print("Phân bố lớp trên tập train SAU SMOTE:")
print("-" * 60)
print(f"  {'Lớp':<25} {'Trước SMOTE':>15} {'Sau SMOTE':>15}")
print("-" * 60)
for idx, name in enumerate(label_names):
    print(f"  {name:<25} {counts_before.get(idx, 0):>15,d} {counts_after.get(idx, 0):>15,d}")
print("-" * 60)
print(f"  {'TỔNG':<25} {sum(counts_before.values()):>15,d} {sum(counts_after.values()):>15,d}")

# Kiểm tra nhanh: các lớp KHÔNG thuộc danh sách SMOTE phải giữ nguyên số mẫu
for idx, name in enumerate(label_names):
    if idx not in minority_classes:
        assert counts_before.get(idx, 0) == counts_after.get(idx, 0), (
            f"Lớp {name} bị thay đổi số mẫu ngoài ý muốn!"
        )
print("\n✓ Đã kiểm tra: các lớp không nằm trong danh sách SMOTE giữ nguyên số mẫu gốc.")


In [ ]:
# ==============================================================================
# 6. LƯU KẾT QUẢ
# ==============================================================================
X_train_smote = pd.DataFrame(X_res, columns=feature_names)
y_train_smote = pd.Series(y_res, name="label")

X_train_smote.to_csv(data_dir + "X_train_smote.csv", index=False)
y_train_smote.to_csv(data_dir + "y_train_smote.csv", index=False)

print(f"✓ Đã lưu: {data_dir}X_train_smote.csv  (shape={X_train_smote.shape})")
print(f"✓ Đã lưu: {data_dir}y_train_smote.csv  (shape={y_train_smote.shape})")

distribution = {
    "before": {label_names[i]: int(counts_before.get(i, 0)) for i in range(len(label_names))},
    "after": {label_names[i]: int(counts_after.get(i, 0)) for i in range(len(label_names))},
    "smote_classes": [label_names[i] for i in minority_classes],
    "target_count": TARGET_COUNT,
    "total_before": int(sum(counts_before.values())),
    "total_after": int(sum(counts_after.values())),
}

with open(data_dir + "smote_class_distribution.json", "w", encoding="utf-8") as f:
    json.dump(distribution, f, indent=4, ensure_ascii=False)

print(f"✓ Đã lưu: {data_dir}smote_class_distribution.json")
print("\nBước tiếp theo: chạy notebook `train_tabnet_smote.ipynb` để huấn luyện "
      "TabNet trên tập train_smote này (val/test giữ nguyên bản gốc).")
